A baseline is a simple model against which we compare more complex algorithms. If a complex model does not outperform the baseline, then its added complexity is not yet justified.

## Problem Definition

The RetailRocket dataset contains implicit user feedback rather than explicit ratings.

Users interact with items through different types of events:

- `view`
- `addtocart`
- `transaction`

These events provide behavioral signals about user interest, but they do not directly tell us how much a user likes an item.

For this project, we treat the interaction strength as:

| Event | Weight |
|---|---:|
| View | 1 |
| Add to cart | 3 |
| Transaction | 5 |

Therefore, the task is not to predict an explicit rating such as 4.5 out of 5.

Instead, the goal is to rank items according to how likely they are to receive a future interaction from a user.

Formally, for a user $u$, we want to generate a ranked list:

$$
\hat{i}_1, \hat{i}_2, \ldots, \hat{i}_K
$$

where the most promising items are placed at the top of the list.

## Explicit vs. Implicit Feedback

In an explicit-feedback recommendation system, the model may predict a rating:

$$
\hat{r}_{ui}
$$

For example:

> "How many stars would user $u$ give item $i$?"

In an implicit-feedback system, the model does not observe an explicit rating.

Instead, it observes user behavior:

- viewing an item,
- adding an item to the cart,
- purchasing an item.

Therefore, the recommendation task is primarily a ranking problem:

> "Which items is user $u$ most likely to interact with next?"

An important distinction is that the absence of an interaction does **not** necessarily mean that the user dislikes the item.

For example:

$$
r_{ui}=0
$$

means:

> No interaction was observed.

It does not mean:

> The user dislikes the item.

## What Makes a Good Baseline?

A useful baseline should be:

1. Simple to implement.
2. Fast to train and evaluate.
3. Easy to understand.
4. Free from unnecessary model complexity.
5. Strong enough to provide a meaningful reference point.

For an e-commerce recommendation system, item popularity is a natural baseline.

If an item is frequently interacted with by many users, recommending it to other users may already provide reasonable recommendation quality.

More advanced models should therefore be evaluated against popularity-based recommendations.

## Baseline Models

We will consider three increasingly informative baselines.

### 1. Global Mean

The simplest possible baseline assigns the same interaction strength to every user-item pair:

$$
\hat{r}_{ui} = \mu
$$

where $\mu$ is the global mean interaction strength.

This model is useful mainly as a conceptual sanity check.

It cannot produce a meaningful Top-K ranking because every item receives the same score.

---

### 2. Item Mean

For each item, we calculate the average interaction strength among users who interacted with it:

$$
\bar{r}_i =
\frac{\sum_{u:r_{ui}>0} r_{ui}}
{\left|\{u:r_{ui}>0\}\right|}
$$

Items with a higher average interaction strength are ranked higher.

This baseline captures the fact that some items tend to receive stronger interactions than others.

However, it can be unstable for items with very few interactions.

---

### 3. Item Popularity

We calculate the total interaction strength for each item:

$$
pop(i) = \sum_u r_{ui}
$$

Items with the highest popularity scores are recommended first.

This is our main baseline because popularity is a natural and surprisingly strong baseline for e-commerce recommendation.

## Interaction Matrix

At the simplest level, implicit feedback can be represented as:

$$
R_{ui} =
\begin{cases}
1, & \text{if an interaction was observed}\\
0, & \text{if no interaction was observed}
\end{cases}
$$

The value `0` should not be interpreted as a negative rating.

It simply means that no interaction was observed in the available data.

In our dataset, we additionally preserve interaction strength:

$$
r_{ui} \in \{1, 3, 5\}
$$

where:

- `1` represents a view,
- `3` represents an add-to-cart event,
- `5` represents a transaction.

This allows us to compare binary popularity with weighted popularity.

In [1]:
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw")

category_tree_path = DATA_DIR / "category_tree.csv"
events_path = DATA_DIR / "events.csv"
item_properties_p1_path = DATA_DIR / "item_properties_part1.csv"
item_properties_p2_path = DATA_DIR / "item_properties_part2.csv"

In [3]:
category_tree = pd.read_csv(category_tree_path)
events = pd.read_csv(events_path)
item_properties_p1 = pd.read_csv(item_properties_p1_path)
item_properties_p2 = pd.read_csv(item_properties_p2_path)

In [4]:
event_weights = {
    "view": 1,
    "addtocart": 3,
    "transaction": 5
}

events["weight"] = events["event"].map(event_weights)

events = events.dropna(subset=["weight"])

In [5]:
interactions = (
    events
    .groupby(["visitorid", "itemid"])
    .agg(
        weight=("weight", "sum"),
        timestamp=("timestamp", "max")
    )
    .reset_index()
)

interactions["datetime"] = pd.to_datetime(
    interactions["timestamp"],
    unit="ms"
)

interactions.head()

,visitorid,itemid,weight,timestamp,datetime
0,0,67045,1,1442004917175,2015-09-11 20:55:17.175
1,0,285930,1,1442004589439,2015-09-11 20:49:49.439
2,0,357564,1,1442004759591,2015-09-11 20:52:39.591
3,1,72028,1,1439487966444,2015-08-13 17:46:06.444
4,2,216305,2,1438971463170,2015-08-07 18:17:43.170


In [6]:
interactions["user_idx"], user_mapping = pd.factorize(
    interactions["visitorid"]
)

interactions["item_idx"], item_mapping = pd.factorize(
    interactions["itemid"]
)

In [7]:
interactions = interactions[
    [
        "user_idx",
        "item_idx",
        "weight",
        "timestamp",
        "datetime"
    ]
]

In [8]:
print(f"Users: {interactions['user_idx'].nunique():,}")
print(f"Items: {interactions['item_idx'].nunique():,}")
print(f"Interactions: {len(interactions):,}")

Users: 1,407,580
Items: 235,061
Interactions: 2,145,179


In [9]:
interactions["weight"].describe()

count    2.145179e+06
mean     1.391303e+00
std      1.639133e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.080000e+02
Name: weight, dtype: float64

In [10]:
class BaselineModel:
    def __init__(self, n_users, n_items):
        self.n_users = n_users
        self.n_items = n_items

        self.global_mean = None
        self.user_bias = None
        self.item_bias = None

    def fit(self, train):

        # We treat unobserved user-item pairs as zero interaction
        # when computing the global mean.
        self.global_mean = (
            train["weight"].sum()
            / (self.n_users * self.n_items)
        )

        # Average interaction strength contributed by each user
        # relative to the global mean.
        self.user_bias = (
            train.groupby("user_idx")["weight"].sum()
            / self.n_items
            - self.global_mean
        )

        # Average interaction strength contributed by each item
        # relative to the global mean.
        self.item_bias = (
            train.groupby("item_idx")["weight"].sum()
            / self.n_users
            - self.global_mean
        )

        return self

    def predict(self, user_idx, item_idx):

        prediction = self.global_mean

        if user_idx in self.user_bias.index:
            prediction += self.user_bias[user_idx]

        if item_idx in self.item_bias.index:
            prediction += self.item_bias[item_idx]

        return prediction

    def recommend(self, user_idx, train_matrix, k=10):

        # Only items observed during training are candidates.
        # Items unseen during training have no learned item bias.
        scores = np.full(self.n_items, -np.inf)

        known_items = self.item_bias.index
        scores[known_items] = self.item_bias.to_numpy()

        # Exclude items already seen by the user.
        start = train_matrix.indptr[user_idx]
        end = train_matrix.indptr[user_idx + 1]

        seen_items = train_matrix.indices[start:end]
        scores[seen_items] = -np.inf

        # Retrieve Top-K items efficiently.
        top_k_items = np.argpartition(-scores, k)[:k]

        # Sort the selected items by their final score.
        top_k_items = top_k_items[
            np.argsort(-scores[top_k_items])
        ]

        return top_k_items

Important: Unlike explicit ratings, implicit feedback does not contain observed zero ratings. Here, the global mean is computed over the full user-item space, treating unobserved interactions as zero interaction strength. This is a simplifying assumption used for this baseline and should not be interpreted as an actual probability of interaction.


### Interpretation of the Baseline Model

The baseline model follows the same additive structure commonly used for explicit-feedback recommendation:

$$
\hat{r}_{ui} = \mu + b_u + b_i
$$

where:

- $\mu$ is the global mean interaction strength;
- $b_u$ is the user bias;
- $b_i$ is the item bias.

For our implicit-feedback dataset, the interaction strength is defined by the event type:

- `view` → 1
- `addtocart` → 3
- `transaction` → 5

The model assumes that an unobserved user-item interaction has zero interaction strength when calculating the global mean and biases.

This is a simplifying assumption. In implicit-feedback datasets, an unobserved interaction does not necessarily mean that the user dislikes the item. It only means that no interaction was observed.

### An Important Property for Top-K Recommendation

Although the model contains both user and item biases, it does not provide true personalization when used for ranking.

For a fixed user $u$:

$$
\hat{r}_{ui} = \mu + b_u + b_i
$$

The global mean $\mu$ and user bias $b_u$ are constant across all items for this user.

Therefore, when ranking items:

$$
\operatorname{rank}(\hat{r}_{ui})
=
\operatorname{rank}(b_i)
$$

The ranking is determined only by the item bias.

As a result, all users receive the same item ordering, apart from filtering out items they have already interacted with.

This means that, for Top-K recommendation, this baseline is effectively a popularity-based model.

This limitation is important because it motivates more advanced personalized recommendation models.

In the next stage, Implicit ALS will introduce user and item latent factors:

$$
\hat{r}_{ui} = p_u^T q_i
$$

allowing the ranking of items to depend on the specific user.

### Train/Test Split

Since recommender systems are used to predict future user behavior, a random train/test split is not appropriate for this task. Random splitting could allow the model to learn from interactions that happened after the interaction we are trying to predict.

Instead, we use a time-based rolling evaluation strategy.

The dataset contains approximately 137 days of interactions. We use the last 10 days as validation windows, with each test window containing exactly one day.

For each window:

- the training set contains all interactions before the test period;
- the test set contains interactions from the following day;
- the training period expands as we move forward in time;
- only users and items already observed in the training data are evaluated.

This setup represents a warm-start recommendation scenario, where the system has previously observed the user and the item and must predict the user's future interactions.

The resulting evaluation scheme is:

```text
Window 1
TRAIN: ────────────────────────────────
TEST:                                  DAY 1

Window 2
TRAIN: ───────────────────────────────────
TEST:                                      DAY 2

Window 3
TRAIN: ───────────────────────────────────────
TEST:                                            DAY 3

...

Window 10
TRAIN: ───────────────────────────────────────────────────────
TEST:                                                           DAY 10

In [11]:
# Aggregate interactions
interactions = (
    events
    .groupby(["visitorid", "itemid"])
    .agg(
        weight=("weight", "sum"),
        timestamp=("timestamp", "max")
    )
    .reset_index()
)

# Convert timestamp to datetime
interactions["datetime"] = pd.to_datetime(
    interactions["timestamp"],
    unit="ms"
)

# Create global user and item indices
interactions["user_idx"], user_mapping = pd.factorize(
    interactions["visitorid"]
)

interactions["item_idx"], item_mapping = pd.factorize(
    interactions["itemid"]
)

In [12]:
n_windows = 10
test_days = 1

max_date = interactions["datetime"].max()

windows = []

for i in range(n_windows, 0, -1):

    test_start = (
        max_date
        - pd.DateOffset(days=i)
    )

    test_end = (
        test_start
        + pd.DateOffset(days=test_days)
    )

    train = interactions[
        interactions["datetime"] < test_start
    ].copy()

    test = interactions[
        (interactions["datetime"] >= test_start)
        & (interactions["datetime"] < test_end)
    ].copy()

    # Keep only users known during training
    known_users = train["user_idx"].unique()

    # Keep only items known during training
    known_items = train["item_idx"].unique()

    test = test[
        test["user_idx"].isin(known_users)
        & test["item_idx"].isin(known_items)
    ].copy()

    windows.append({
        "iteration": n_windows - i + 1,
        "train": train,
        "test": test,
        "train_end": test_start,
        "test_start": test_start,
        "test_end": test_end,
    })

In [13]:
for window in windows:

    train = window["train"]
    test = window["test"]

    print(
        f"Window {window['iteration']:2d} | "
        f"Train: {window['train_end'].date()} | "
        f"Test: {window['test_start'].date()} → "
        f"{window['test_end'].date()} | "
        f"Train interactions: {len(train):,} | "
        f"Test interactions: {len(test):,}"
    )

Window  1 | Train: 2015-09-08 | Test: 2015-09-08 → 2015-09-09 | Train interactions: 1,999,142 | Test interactions: 2,880
Window  2 | Train: 2015-09-09 | Test: 2015-09-09 → 2015-09-10 | Train interactions: 2,015,790 | Test interactions: 3,319
Window  3 | Train: 2015-09-10 | Test: 2015-09-10 → 2015-09-11 | Train interactions: 2,033,186 | Test interactions: 3,433
Window  4 | Train: 2015-09-11 | Test: 2015-09-11 → 2015-09-12 | Train interactions: 2,051,702 | Test interactions: 2,959
Window  5 | Train: 2015-09-12 | Test: 2015-09-12 → 2015-09-13 | Train interactions: 2,067,832 | Test interactions: 2,041
Window  6 | Train: 2015-09-13 | Test: 2015-09-13 → 2015-09-14 | Train interactions: 2,080,882 | Test interactions: 1,965
Window  7 | Train: 2015-09-14 | Test: 2015-09-14 → 2015-09-15 | Train interactions: 2,094,902 | Test interactions: 2,889
Window  8 | Train: 2015-09-15 | Test: 2015-09-15 → 2015-09-16 | Train interactions: 2,112,076 | Test interactions: 2,988
Window  9 | Train: 2015-09-16 | 

#### Evaluation Metrics

For implicit-feedback recommendation, the main objective is not to predict an exact numerical rating. Instead, the model should rank items that the user is likely to interact with higher than other items.

Therefore, we evaluate the quality of the Top-K recommendation list rather than the reconstruction loss used during ALS training.

For each user:

1. The model computes a relevance score for candidate items.
2. Items are sorted by their predicted score.
3. The top K items are selected as recommendations.
4. The recommendations are compared with the items the user actually interacted with during the test period.

Only interactions observed in the test period are considered relevant.

A particularly important aspect of our evaluation is how much of the user's future behavior we are able to cover. If a user interacts with three items during the test period and two of them appear in the Top-K recommendations, the model covers 2 out of 3 relevant items, achieving a Recall@K of 66.7%.

This makes Recall@K especially useful for our dataset, where many users have only a small number of interactions.

We use three ranking metrics:

- Precision@K measures the proportion of recommended items that were relevant.
- Recall@K measures the proportion of relevant test items covered by the recommendations.
- NDCG@K measures how well relevant items are ranked, giving higher importance to relevant items appearing near the top of the recommendation list.

Items already observed by the user during training are excluded from the recommendation candidates. This ensures that the evaluation measures the ability of the model to recommend new items that the user interacts with in the future.

We evaluate the metrics at K = 10, which represents the Top-10 recommendation list.

In [14]:
def precision_at_k(recommended_items, relevant_items, k):
    
    recommended_items = recommended_items[:k]
    
    if len(recommended_items) == 0:
        return 0.0
    
    hits = len(
        set(recommended_items) & set(relevant_items)
    )
    
    return hits / k

In [15]:
def recall_at_k(recommended_items, relevant_items, k):
    
    relevant_items = set(relevant_items)
    
    if len(relevant_items) == 0:
        return 0.0
    
    recommended_items = set(
        recommended_items[:k]
    )
    
    hits = len(
        recommended_items & relevant_items
    )
    
    return hits / len(relevant_items)

In [16]:
import numpy as np


def ndcg_at_k(recommended_items, relevant_items, k):
    
    relevant_items = set(relevant_items)
    
    if len(relevant_items) == 0:
        return 0.0
    
    recommended_items = recommended_items[:k]
    
    dcg = 0.0
    
    for rank, item in enumerate(recommended_items, start=1):
        
        if item in relevant_items:
            dcg += 1 / np.log2(rank + 1)
    
    ideal_length = min(len(relevant_items), k)
    
    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(1, ideal_length + 1)
    )
    
    return dcg / idcg

In [20]:
n_users = interactions["user_idx"].nunique()
n_items = interactions["item_idx"].nunique()

results = []

for window in windows:

    train = window["train"]
    test = window["test"]

    train_matrix = csr_matrix(
        (
            train["weight"],
            (
                train["user_idx"],
                train["item_idx"]
            )
        ),
        shape=(n_users, n_items),
        dtype=np.float64
    )

    model = BaselineModel(
        n_users=n_users,
        n_items=n_items
    )

    model.fit(train)

    test_relevant = (
        test
        .groupby("user_idx")["item_idx"]
        .apply(set)
        .to_dict()
    )

    precision_scores = []
    recall_scores = []
    ndcg_scores = []

    for user_idx, relevant_items in test_relevant.items():

        recommendations = model.recommend(
            user_idx=user_idx,
            train_matrix=train_matrix,
            k=10
        )

        precision_scores.append(
            precision_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

        recall_scores.append(
            recall_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommendations,
                relevant_items,
                k=10
            )
        )

    results.append({
        "window": window["iteration"],
        "precision@10": np.mean(precision_scores),
        "recall@10": np.mean(recall_scores),
        "ndcg@10": np.mean(ndcg_scores),
    })

In [21]:
results_df = pd.DataFrame(results)

results_df

,window,precision@10,recall@10,ndcg@10
0,1,0.001220,0.006275,0.003729
1,2,0.001122,0.006883,0.003724
2,3,0.000636,0.003203,0.001423
3,4,0.001022,0.007436,0.004098
4,5,0.001397,0.009122,0.006579
5,6,0.001096,0.006913,0.004791
6,7,0.000765,0.004428,0.003128
7,8,0.000494,0.002952,0.001765
8,9,0.001430,0.005225,0.003093
9,10,0.000987,0.003809,0.003919


### Baseline Results

The baseline provides a simple reference point for evaluating more advanced
recommendation models.

Because the additive model is composed of a global mean, user bias, and item
bias, the user-specific terms are constant across candidate items for a given
user. Therefore, they do not affect the Top-K ranking.

As a result, this baseline effectively ranks items primarily by their
historical popularity.

The relatively low Recall@10 and NDCG@10 show the limitation of popularity-based
recommendation: users do not necessarily interact with the globally strongest
items.

This motivates moving to collaborative filtering methods such as ALS, which can
model user-item interactions through latent representations.